# AEM Final

In [ ]:
import requests
import pandas as pd

url = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
params = {
    "$limit": 5
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()
df = pd.DataFrame(data)

print(df)
print(df.columns.tolist())

In [ ]:
import requests
import pandas as pd

url = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
select_columns = [
    "id",
    "case_number",
    "date",
    "primary_type",
    "description",
    "location_description",
    "arrest",
    "domestic",
    "beat",
    "district",
    "ward",
    "community_area",
    "latitude",
    "longitude"
]

where_clause = "date >= '2024-01-01T00:00:00' AND date < '2025-01-01T00:00:00'"
page_size = 50000
offset = 0
records = []

while True:
    params = {
        "$select": ",".join(select_columns),
        "$where": where_clause,
        "$order": "date ASC, id ASC",
        "$limit": page_size,
        "$offset": offset
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    page = response.json()

    if not page:
        break

    records.extend(page)
    print(f"Downloaded {len(records):,} records...")

    if len(page) < page_size:
        break

    offset += page_size

df = pd.DataFrame(records)

print("Shape before cleaning:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

for column in select_columns:
    if column not in df.columns:
        df[column] = pd.NA

df = df[select_columns]

df["date"] = pd.to_datetime(df["date"], errors="coerce")
for column in ["latitude", "longitude", "beat", "district", "ward", "community_area"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in ["arrest", "domestic"]:
    df[column] = df[column].map({"true": True, "false": False, True: True, False: False})

df = df.dropna(subset=["date"])

for column in ["primary_type", "description", "location_description"]:
    df[column] = df[column].astype("string").str.strip().str.upper()

df["hour"] = df["date"].dt.hour
df["day_of_week"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()
df["year"] = df["date"].dt.year
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"])

print("\nShape after cleaning:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nPreview:")
print(df.head())

df.to_csv("chicago_crimes_2024_clean.csv", index=False)
print("\nSaved file: chicago_crimes_2024_clean.csv")

df_map = df.dropna(subset=["latitude", "longitude"]).copy()
df_map.to_csv("crimes_map_ready.csv", index=False)
print("Saved file: crimes_map_ready.csv")
print("Map-ready shape:", df_map.shape)



In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())
print(df.head())
print(df.describe(include="all"))

In [ ]:
df_map = df.dropna(subset=["latitude", "longitude"]).copy()
df_map.to_csv("crimes_map_ready.csv", index=False)
print("Map-ready shape:", df_map.shape)


In [ ]:
print("Cleaned dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

print("\nPreview:")
display(df.head())

We retrieved the Chicago Crimes dataset through the Socrata Open Data API and selected key variables related to offense type, location, arrest status, and geography. The final dataset is filtered to full-year 2024 records using `date >= '2024-01-01T00:00:00' AND date < '2025-01-01T00:00:00'`. Because the API returns data in pages, we used pagination with `$limit`, `$offset`, and stable ordering by `date` and `id` to avoid stopping at the first 50,000 records. We cleaned the data by converting the date column to datetime format, converting geographic and district fields to numeric format, standardizing text fields, and creating additional time-based variables including hour, day of week, month, month name, year, and a weekend indicator. We also exported a map-ready dataset after removing records without latitude or longitude.


This section examines temporal crime patterns in the cleaned Chicago crimes dataset. We focus on how incident counts vary by hour, day of week, and month, and we also compare arrest rates across time periods. The goal is to identify whether crime incidents are evenly distributed over time or concentrated in specific recurring periods.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("chicago_crimes_2024_clean.csv")
df["date"] = pd.to_datetime(df["date"])

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nCrime count by hour:")
print(df["hour"].value_counts().sort_index())

print("\nCrime count by day of week:")
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(df["day_of_week"].value_counts().reindex(weekday_order))

print("\nCrime count by month:")
month_order = ["January", "February", "March", "April", "May", "June", "July",
               "August", "September", "October", "November", "December"]
print(df["month_name"].value_counts().reindex(month_order).dropna())

print("\nArrest rate by hour:")
arrest_by_hour = df.groupby("hour")["arrest"].mean().sort_index()
print(arrest_by_hour)

print("\nArrest rate by day of week:")
arrest_by_day = df.groupby("day_of_week")["arrest"].mean().reindex(weekday_order)
print(arrest_by_day)

top5_crimes = df["primary_type"].value_counts().head(5).index.tolist()
df_top5 = df[df["primary_type"].isin(top5_crimes)].copy()

plt.figure(figsize=(10, 5))
df["hour"].value_counts().sort_index().plot(kind="bar")
plt.title("Crime Count by Hour")
plt.xlabel("Hour")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
df["day_of_week"].value_counts().reindex(weekday_order).plot(kind="bar")
plt.title("Crime Count by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

month_counts = df["month_name"].value_counts().reindex(month_order).dropna()
plt.figure(figsize=(10, 5))
month_counts.plot(kind="bar")
plt.title("Crime Count by Month")
plt.xlabel("Month")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

heatmap_data = df.groupby(["day_of_week", "hour"]).size().unstack(fill_value=0).reindex(weekday_order)
plt.figure(figsize=(12, 5))
sns.heatmap(heatmap_data, cmap="YlOrRd")
plt.title("Crime Count Heatmap: Day of Week × Hour")
plt.xlabel("Hour")
plt.ylabel("Day of Week")
plt.tight_layout()
plt.show()

top5_hourly = df_top5.groupby(["hour", "primary_type"]).size().unstack(fill_value=0)
plt.figure(figsize=(10, 5))
top5_hourly.plot(ax=plt.gca())
plt.title("Top 5 Crime Types by Hour")
plt.xlabel("Hour")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
arrest_by_hour.plot(marker="o")
plt.title("Arrest Rate by Hour")
plt.xlabel("Hour")
plt.ylabel("Arrest Rate")
plt.tight_layout()
plt.show()

domestic_time = df.groupby(["day_of_week", "domestic"]).size().unstack(fill_value=0).reindex(weekday_order)
plt.figure(figsize=(10, 5))
domestic_time.plot(kind="bar", ax=plt.gca())
plt.title("Domestic vs Non-Domestic Crimes by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## Temporal Analysis

This section should be rerun after the full-year 2024 data update. The cleaned dataset now covers incidents from **2024-01-01 00:00:00** through **2024-12-31 23:58:00**, so month-level charts can now be interpreted as full-year 2024 patterns instead of a partial January-March sample.

Initial validation confirms that all 12 months are present in `chicago_crimes_2024_clean.csv`. The current monthly incident counts are: January 19,653; February 19,948; March 20,914; April 20,491; May 22,990; June 23,205; July 24,099; August 23,000; September 22,980; October 22,492; November 19,734; December 19,605.

The temporal analysis code above should now be rerun so the hourly, weekday, monthly, heatmap, top-crime-type, and arrest-rate charts use the full 2024 dataset.
